In [3]:
from json.decoder import NaN

import ltatools as lta
import numpy as np
import matplotlib.pyplot as plt
import allantools as at
import pandas as pd

In [10]:
df0 = lta.load_lta_file('data/06.07.2026, 15.12,  268,0962819 THz.lta', cleanup=False)
df1 = lta.load_lta_file('data/06.07.2026, 14.55,  268,0962734 THz.lta', cleanup=False)
df2 = lta.load_lta_file('data/06.07.2026, 14.45,  268,0962640 THz.lta', cleanup=False)

for i in range(int(df0['time_s'].max()) // 30):
    lim_lo = i * 30
    lim_hi = lim_lo + 30

    df = df0[(df0['time_s'] > lim_lo) & (df0['time_s'] < lim_hi)].reset_index(drop=True)

    #lta.plot(data=df, kind='adev', quantity='frequency', errorbars=True, unit='MHz', regions=True, print_regions=True, figsize=(7, 4), label='')
    #lta.plot(data=df, kind='timeseries', lines=True, relative=True, figsize=(9, 4), label='', tick_direction='in')

for i in range(int(df1['time_s'].max()) // 30):
    lim_lo = i * 30
    lim_hi = lim_lo + 30

    df = df1[(df1['time_s'] > lim_lo) & (df1['time_s'] < lim_hi)].reset_index(drop=True)

    #lta.plot(data=df, kind='adev', quantity='frequency', errorbars=True, unit='MHz', regions=True, print_regions=True, figsize=(7, 4), label='')
    #lta.plot(data=df, kind='timeseries', lines=True, relative=True, figsize=(9, 4), label='', tick_direction='in')

for i in range(int(df2['time_s'].max()) // 30):
    lim_lo = i * 30
    lim_hi = lim_lo + 30

    df = df2[(df2['time_s'] > lim_lo) & (df2['time_s'] < lim_hi)].reset_index(drop=True)

    #lta.plot(data=df, kind='adev', quantity='frequency', errorbars=True, unit='MHz', regions=True, print_regions=True, figsize=(7, 4), label='')
    #lta.plot(data=df, kind='timeseries', lines=True, relative=True, figsize=(9, 4), label='', tick_direction='in')

In [40]:
df5_noIso = lta.load_lta_file('data/02.07.2026, 18.13,  268,0962595 THz.lta', cleanup=False)
df6_noIso = lta.load_lta_file('data/02.07.2026, 19.15,  268,0959664 THz.lta', cleanup=False)

#lta.plot(data=df5_noIso, kind='timeseries', lines=True, relative=True, freq_unit='MHz', figsize=(9, 4), label='', tick_direction='in')

for i in range(int(df5_noIso['time_s'].max()) // 30):
    lim_lo = i * 30
    lim_hi = lim_lo + 30

    df = df5_noIso[(df5_noIso['time_s'] > lim_lo) & (df5_noIso['time_s'] < lim_hi)].reset_index(drop=True)

    #lta.plot(data=df, kind='adev', quantity='frequency', errorbars=True, unit='MHz', regions=True, print_regions=True, figsize=(7, 4), label='')
    #lta.plot(data=df, kind='timeseries', lines=True, relative=True, figsize=(9, 4), label='', tick_direction='in')

#for i in range(int(df6_noIso['time_s'].max()) // 30):
#    lim_lo = i * 30
#    lim_hi = lim_lo + 30
#
#    df = df6_noIso[(df6_noIso['time_s'] > lim_lo) & (df6_noIso['time_s'] < lim_hi)].reset_index(drop=True)

#    lta.plot(data=df, kind='adev', quantity='frequency', errorbars=True, unit='MHz', regions=True, print_regions=True, figsize=(7, 4), label='')
#    lta.plot(data=df, kind='timeseries', lines=True, relative=True, figsize=(9, 4), label='', tick_direction='in')

In [41]:
from ltatools.analysis import summarize_adev_regions, DEFAULT_ADEV_REGION_BOUNDARIES
from ltatools.style import finer_unit, scale_frequency

def shortest_term_region_df(datasets, window_s=30, unit='MHz'):
    region_unit = finer_unit(unit, 'frequency')
    rows = []
    for name, dataframe in datasets.items():
        for i in range(int(dataframe['time_s'].max()) // window_s):
            lim_lo = i * window_s
            lim_hi = lim_lo + window_s
            window_df = dataframe[(dataframe['time_s'] > lim_lo) & (dataframe['time_s'] < lim_hi)].reset_index(drop=True)

            tau, dev, dev_err, _ = lta.compute_oadev(window_df['frequency_THz'], time_s=window_df['time_s'])
            dev_scaled = scale_frequency(dev, region_unit)
            dev_err_scaled = scale_frequency(dev_err, region_unit)
            regions = summarize_adev_regions(tau, dev_scaled, dev_err_scaled, boundaries=DEFAULT_ADEV_REGION_BOUNDARIES)
            short = regions[0]  # shortest-term region: tau in [0, first boundary) s

            rows.append({
                'dataset': name, 'window': i, 'lim_lo_s': lim_lo, 'lim_hi_s': lim_hi,
                'value': short['value'], 'error': short['error'], 'unit': region_unit, 'n': short['n'],
            })

    return pd.DataFrame(rows)

In [42]:
datasets = {'df0': df0, 'df1': df1, 'df2': df2}
shortterm_regions_0 = shortest_term_region_df(datasets)
#shortterm_regions_0 = shortterm_regions_0.drop(index=[2, 5, 7]).reset_index(drop=True)
mean_shortterm_0 = shortterm_regions_0['value'].mean()
std_shortterm_0 = shortterm_regions_0['value'].std()
print(f"Mean of shortest-term regions (post-mod, insulated): {mean_shortterm_0:.2f} {shortterm_regions_0['unit'].iloc[0]}")
print(f"Standard deviation of shortest-term regions (post-mod, insulated): {std_shortterm_0:.2f} {shortterm_regions_0['unit'].iloc[0]}")

#shortterm_regions_0

Mean of shortest-term regions (post-mod, insulated): 86.26 kHz
Standard deviation of shortest-term regions (post-mod, insulated): 6.95 kHz


In [43]:
datasets_noIso = {'df5_noIso': df5_noIso}
shortterm_regions_noIso = shortest_term_region_df(datasets_noIso)
#shortterm_regions_noIso = shortterm_regions_noIso.drop(index=[4, 7]).reset_index(drop=True)
mean_shortterm_noIso = shortterm_regions_noIso['value'].mean()
std_shortterm_noIso = shortterm_regions_noIso['value'].std()
print(f"Mean of shortest-term regions (post-mod, no insulation): {mean_shortterm_noIso:.2f} {shortterm_regions_noIso['unit'].iloc[0]}")
print(f"Standard deviation of shortest-term regions (post-mod, no insulation): {std_shortterm_noIso:.2f} {shortterm_regions_noIso['unit'].iloc[0]}")

#shortterm_regions_noIso

Mean of shortest-term regions (post-mod, no insulation): 137.68 kHz
Standard deviation of shortest-term regions (post-mod, no insulation): 32.76 kHz


In [44]:
datasets_noIso = {'df5_noIso': df5_noIso}
shortterm_regions_noIso = shortest_term_region_df(datasets_noIso)
#shortterm_regions_noIso = shortterm_regions_noIso.drop(index=[4, 7]).reset_index(drop=True)
mean_shortterm_noIso = shortterm_regions_noIso['value'].mean()
std_shortterm_noIso = shortterm_regions_noIso['value'].std()
print(f"Mean of shortest-term regions (post-mod, no insulation): {mean_shortterm_noIso:.2f} {shortterm_regions_noIso['unit'].iloc[0]}")
print(f"Standard deviation of shortest-term regions (post-mod, no insulation): {std_shortterm_noIso:.2f} {shortterm_regions_noIso['unit'].iloc[0]}")

#shortterm_regions_noIso

datasets = {'df0': df0, 'df1': df1, 'df2': df2}
shortterm_regions_0 = shortest_term_region_df(datasets)
#shortterm_regions_0 = shortterm_regions_0.drop(index=[2, 5, 7]).reset_index(drop=True)
mean_shortterm_0 = shortterm_regions_0['value'].mean()
std_shortterm_0 = shortterm_regions_0['value'].std()
print(f"Mean of shortest-term regions (post-mod, insulated): {mean_shortterm_0:.2f} {shortterm_regions_0['unit'].iloc[0]}")
print(f"Standard deviation of shortest-term regions (post-mod, insulated): {std_shortterm_0:.2f} {shortterm_regions_0['unit'].iloc[0]}")

#shortterm_regions_0

Mean of shortest-term regions (post-mod, no insulation): 137.68 kHz
Standard deviation of shortest-term regions (post-mod, no insulation): 32.76 kHz
Mean of shortest-term regions (post-mod, insulated): 86.26 kHz
Standard deviation of shortest-term regions (post-mod, insulated): 6.95 kHz
